# StoryForge AI – AI Story Continuation System

This notebook walks you through the full workflow for the **Project 3** assignment: 
* loading the DistilGPT2 model,
* running the Streamlit UI locally,
* packaging the model for TorchServe,
* starting TorchServe and testing the REST API,
* deploying the model to Amazon SageMaker, and
* cleaning up SageMaker resources.

You can execute the cells sequentially, or copy the commands into a terminal if you prefer.  All commands assume you are in the project root (`AI-Story-Continuation`).

## 1️⃣ Environment setup

In [ ]:
# Create a virtual environment (only needed once)
!python3 -m venv .venv

In [ ]:
# Activate the environment and upgrade pip
!source .venv/bin/activate && pip install --upgrade pip

In [ ]:
# Install project dependencies
!source .venv/bin/activate && pip install -r requirements.txt

## 2️⃣ Run the Streamlit UI locally

In [ ]:
# Launch the app (it will open http://localhost:8501)
!source .venv/bin/activate && streamlit run app.py

## 3️⃣ Package the model for TorchServe

In [ ]:
# This script downloads DistilGPT2 (if not already cached) and creates a .mar archive
!bash scripts/package_model.sh

## 4️⃣ Start TorchServe (local)

In [ ]:
# Start TorchServe with the packaged model
!torchserve --start \
    --model-store model-store \
    --models storymodel=storymodel.mar \
    --ts-config torchserve/config.properties

## 5️⃣ Test the TorchServe REST endpoint

In [ ]:
# Use the provided curl script (you can also run it in a terminal)
!bash scripts/test_torchserve.sh

## 6️⃣ Deploy to Amazon SageMaker (cloud)
*You need a faculty‑approved AWS account and the appropriate IAM role.*

In [ ]:
# Replace the placeholders with your own values
ROLE_ARN='arn:aws:iam::YOUR_ACCOUNT_ID:role/YourSageMakerRole'
BUCKET='your-sagemaker-bucket'
REGION='us-east-1'

# Deploy – this will upload the model, create the endpoint and invoke a test request
!source .venv/bin/activate && python sagemaker/deploy_sagemaker.py \
    --role $ROLE_ARN \
    --bucket $BUCKET \
    --region $REGION

## 7️⃣ Clean up SageMaker resources (VERY IMPORTANT)

In [ ]:
# Delete the endpoint, config, model and S3 object to avoid charges
!source .venv/bin/activate && python sagemaker/cleanup.py

---
### 📚 Quick recap
* **Step 1** – set up a Python venv and install dependencies.
* **Step 2** – run `streamlit run app.py` to experiment locally.
* **Step 3** – `bash scripts/package_model.sh` builds a TorchServe `.mar` archive.
* **Step 4** – start TorchServe and hit the REST API (`scripts/test_torchserve.sh`).
* **Step 5** – `python sagemaker/deploy_sagemaker.py …` spins up a SageMaker endpoint.
* **Step 6** – after demonstrating, run `python sagemaker/cleanup.py` to delete the endpoint.

You now have a complete end‑to‑end pipeline that satisfies all rubric items for the assignment.